## ライブラリインポート

In [ ]:
!pip install keras-core --upgrade
!pip install -q keras-nlp --upgrade

# This sample uses Keras Core, the multi-backend version of Keras.
# The selected backend is TensorFlow (other supported backends are 'jax' and 'torch')
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
import keras
import keras_nlp
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import re

print("TensorFlow version:", tf.__version__)
print("KerasNLP version:", keras_nlp.__version__)

##　データ読み込み

In [ ]:
df_train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
df_test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

### データ確認

In [ ]:
df_train.head(5)

In [ ]:
df_test.head(5)

In [ ]:
df_train["length"] = df_train["text"].apply(lambda x : len(x))
df_test["length"] = df_test["text"].apply(lambda x : len(x))

print("Train Length Stat")
print(df_train["length"].describe())
print()

print("Test Length Stat")
print(df_test["length"].describe())

In [ ]:
BATCH_SIZE = 32
NUM_TRAINING_EXAMPLES = df_train.shape[0]
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.2
STEPS_PER_EPOCH = int(NUM_TRAINING_EXAMPLES)*TRAIN_SPLIT // BATCH_SIZE

EPOCHS = 2
AUTO = tf.data.experimental.AUTOTUNE

## データの前処理

In [ ]:
def clean_text(text):
    text = text.lower() #小文字化
    text = re.sub(r'https?://\S+|www\.\S+','',text) #URLの削除
    text = re.sub(r'<.*?>','',text) #HTMLタグの削除
    text = re.sub(r'[^a-z0-9\s]','',text) #英数字とスペース以外の文字を削除
    return text

In [ ]:
df_train['clean_text'] = df_train['text'].apply(clean_text)

In [ ]:
#災害の発生を示唆する単語の有無で特徴量を作成
disaster_keywords = ['fire','earthquake','help','dead','explosion','emergency','flood','injured','rescue','hit']

df_train['disaster_word'] = df_train['text'].str.lower().apply(
    lambda x: int(any(keyword in x for keyword in disaster_keywords))
)

In [ ]:
df_train['disaster_word'] = df_train['disaster_word'].astype(object)

In [ ]:
print(df_train.dtypes)

In [ ]:
df_train.head(5)

## 機械学習の準備

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# 特徴量ベクトル化（clean_text → TF-IDF）
vectorizer = TfidfVectorizer(max_features=1000)
text_vec = vectorizer.fit_transform(df_train['clean_text']).toarray()

# disaster_wordをfloat型に変換して結合
df_train['disaster_word'] = df_train['disaster_word'].astype(np.float32)
X_numeric = df_train[['disaster_word']].values
X_all = np.hstack((text_vec, X_numeric))

# 目的変数
y = df_train['target'].astype(np.int32).values

# 分割
X_train, X_val, y_train, y_val = train_test_split(X_all, y, test_size=VAL_SPLIT, random_state=42)

# テストデータ処理
df_test['clean_text'] = df_test['text'].apply(clean_text)
text_vec_test = vectorizer.transform(df_test['clean_text']).toarray()
df_test['disaster_word'] = df_test['text'].str.lower().apply(
    lambda x: int(any(keyword in x for keyword in ['fire', 'earthquake', 'tsunami', 'hit']))
).astype(np.float32)
X_test_numeric = df_test[['disaster_word']].values
X_test = np.hstack((text_vec_test, X_test_numeric))

In [ ]:
# DistilBERTをロード
preset= "distil_bert_base_en_uncased"

# 設定
preprocessor = keras_nlp.models.DistilBertPreprocessor.from_preset(preset,
                                                                   sequence_length=160,
                                                                   name="preprocessor_4_tweets"
                                                                  )

# Pretrained classifier
classifier = keras_nlp.models.DistilBertClassifier.from_preset(preset,
                                                               preprocessor = preprocessor, 
                                                               num_classes=2)

classifier.summary()

In [ ]:
# Compile
# DataFrameベースで分割し、clean_text（文字列）列のみを学習に使用
df_train_model, df_val_model = train_test_split(df_train, test_size=VAL_SPLIT, random_state=42)

X_train = df_train_model["clean_text"].astype(str)
y_train = df_train_model["target"].astype(int)

X_val = df_val_model["clean_text"].astype(str)
y_val = df_val_model["target"].astype(int)

# DistilBERT は文字列入力しか受け付けないため、ベクトルではなく生のテキストを渡す
classifier.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(1e-5),
    metrics=["accuracy"]
)

#学習
history = classifier.fit(
    x=X_train,
    y=y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val)
)

### モデルのテスト

In [ ]:
def displayConfusionMatrix(y_true, y_pred, dataset):
    disp = ConfusionMatrixDisplay.from_predictions(
        y_true,
        np.argmax(y_pred, axis=1),
        display_labels=["Not Disaster","Disaster"],
        cmap=plt.cm.Blues
    )

    tn, fp, fn, tp = confusion_matrix(y_true, np.argmax(y_pred, axis=1)).ravel()
    f1_score = tp / (tp+((fn+fp)/2))

    disp.ax_.set_title("Confusion Matrix on " + dataset + " Dataset -- F1 Score: " + str(f1_score.round(2)))

## テストデータについて結果を出力

In [ ]:
y_pred_train = classifier.predict(X_train)

displayConfusionMatrix(y_train, y_pred_train, "Training")

In [ ]:
y_pred_val = classifier.predict(X_val)

displayConfusionMatrix(y_val, y_pred_val, "Validation")

## 提出用ファイル作成

In [ ]:
sample_submission = pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")
sample_submission.head()

In [ ]:
# 生のテキスト（str型）を予測対象として準備
X_test_text = df_test["clean_text"].astype(str)

# 予測実行
y_pred_test = classifier.predict(X_test_text)

# 分類ラベルを取得（shape: [n_samples, 2] → クラスラベル）
y_pred_test_class = np.argmax(y_pred_test, axis=1)

# 提出ファイル作成
sample_submission["target"] = y_pred_test_class
print(sample_submission.describe())
sample_submission.to_csv("submission.csv", index=False)